# Fit a new video with PyTorch

Run this notebook in the installed project environment. The detector downloads its pretrained weights on first use. Fitting uses PyTorch autograd and MuJoCo model metadata.


In [ ]:
import os
import sys
import torch
import numpy as np
import cv2
if sys.platform != "darwin":
    os.environ.setdefault("MUJOCO_GL", "egl")
    os.environ.setdefault("PYOPENGL_PLATFORM", "egl")
from monocular_demos.utils import load_metrabs, video_reader, joint_names
from monocular_demos.torch_dataset import MonocularDataset, get_samsung_calibration
from monocular_demos.biomechanics_mjx.torch_monocular_trajectory import get_model, fit_model
video_path = "data/running_sagittal.mp4"


In [ ]:
detector = load_metrabs()
video, n_frames = video_reader(video_path, batch_size=2)
kp2d, kp3d, confidence = [], [], []
for batch in video:
    prediction = detector.detect_poses_batched(batch, skeleton="bml_movi_87")
    for i in range(len(batch)):
        found = len(prediction["boxes"][i]) > 0
        kp2d.append(prediction["poses2d"][i][0].numpy() if found else np.zeros((87, 2)))
        kp3d.append(prediction["poses3d"][i][0].numpy() if found else np.zeros((87, 3)))
        confidence.append(np.ones(87) if found else np.zeros(87))
cap = cv2.VideoCapture(video_path)
fps = cap.get(cv2.CAP_PROP_FPS)
cap.release()
dataset = MonocularDataset(
    timestamps=[np.arange(len(kp2d)) / fps],
    keypoints_2d=[np.stack(kp2d)[None]], keypoints_3d=[np.stack(kp3d)[None]],
    keypoint_confidence=[np.stack(confidence)[None]], camera_params=get_samsung_calibration(),
)


In [ ]:
model = get_model(dataset, joint_names=joint_names)
model, metrics = fit_model(model, dataset, max_iters=10000, lr_init_value=1e-3)
{k: v.mean().item() for k, v in metrics.items()}


In [ ]:
with torch.no_grad():
    (state, _, _), (qpos, qvel, _), rnc = model(dataset.get_all_timestamps(0))
    np.savez("reconstruction_fitted_model.npz", timestamps=dataset.get_all_timestamps(0),
             qpos=qpos.cpu().numpy(), qvel=qvel.cpu().numpy(), rnc=rnc.cpu().numpy(),
             sites=state.site_xpos.cpu().numpy(), joints=state.xpos.cpu().numpy(),
             scale=model.body_scale.cpu().numpy(), video_path=video_path)


In [ ]:
from monocular_demos.biomechanics_mjx.visualize import render_trajectory, jupyter_embed_video
render_trajectory(qpos.cpu().numpy(), "reconstruction.mp4", body_scale=model.body_scale.detach().cpu().numpy())
jupyter_embed_video("reconstruction.mp4")
